#  Course Recommendation System


---
## Importing Libraries

In [21]:
import pandas as pd
import numpy as np
import re
import pickle
import warnings
warnings.filterwarnings('ignore')

# NLP & ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler



---
## 2.  Loadind & Inspecting Dataset

In [22]:
df = pd.read_csv('udemy_courses_cleaned.csv')
df.head()

,published,course_id,course_title,url,is_paid,price,num_subscribers,num_reviews,num_lectures,level,content_duration,subject,published.1,year,quarter,month,profit,content_duration_cat,price_cat
0,2017-01-18,1070968,Ultimate Investment Banking Course,https://www.udemy.com/ultimate-investment-bank...,True,200,2147,23,51,All Levels,1.5,Business Finance,2017-01-18,2017,1,January,429400,1 : 3,170:200
1,2016-12-19,1006314,Financial Modeling for Business Analysts and C...,https://www.udemy.com/financial-modeling-for-b...,True,45,2174,74,51,Intermediate Level,2.5,Business Finance,2016-12-19,2016,4,December,97830,1 : 3,20:50
2,2017-05-30,1210588,Beginner to Pro - Financial Analysis in Excel ...,https://www.udemy.com/complete-excel-finance-c...,True,95,2451,11,36,All Levels,3.0,Business Finance,2017-05-30,2017,2,May,232845,1 : 3,80:110
3,2016-12-13,1011058,How To Maximize Your Profits Trading Options,https://www.udemy.com/how-to-maximize-your-pro...,True,200,1276,45,26,Intermediate Level,2.0,Business Finance,2016-12-13,2016,4,December,255200,1 : 3,170:200
4,2015-05-28,476268,Options Trading 3 : Advanced Stock Profit and ...,https://www.udemy.com/day-trading-stock-option...,True,195,5172,34,38,Expert Level,2.5,Business Finance,2015-05-28,2015,2,May,1008540,1 : 3,170:200


---
## 3.  Feature Engineering for the Recommender

In [23]:
def clean_text(text):
    """
    Normalize course title text:
    - Lowercase
    - Remove special characters & extra spaces
    - Keep alphanumeric and spaces
    """
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Apply text cleaning
df['clean_title'] = df['course_title'].apply(clean_text)



In [24]:
def build_metadata_soup(row):
    """
    Combine course title with metadata fields into one enriched text string.
    Repeating the title 3x gives it higher weight vs metadata.
    Subject is repeated 2x to emphasize domain relevance.
    """
    title   = clean_text(row['course_title'])
    subject = clean_text(row['subject'])       # e.g. 'business finance'
    level   = clean_text(row['level'])         # e.g. 'beginner level'

    soup = f"{title} {title} {title} {subject} {subject} {level}"
    return soup


df['soup'] = df.apply(build_metadata_soup, axis=1)



---
## 4.  Build TF-IDF Models

We build **two models**:

| Model | Input | Best for |
|-------|-------|----------|
| **Title-only** | `clean_title` | Exact keyword matching on titles |
| **Enriched** | `soup` (title + subject + level) | Semantically broader, context-aware suggestions |

In [25]:
# ── Model 1: Title-only ──────────────────────────────────────────────────────
tfidf_title = TfidfVectorizer(
    analyzer      = 'word',
    ngram_range   = (1, 2),   # unigrams + bigrams
    min_df        = 1,
    stop_words    = 'english',
    sublinear_tf  = True       # dampen high-frequency term weight
)

tfidf_matrix_title = tfidf_title.fit_transform(df['clean_title'])
cosine_sim_title   = cosine_similarity(tfidf_matrix_title, tfidf_matrix_title)

print(f'   Title-only model')
print(f'   Vocabulary size : {len(tfidf_title.vocabulary_):,} terms')
print(f'   TF-IDF matrix   : {tfidf_matrix_title.shape}')
print(f'   Cosine sim mat  : {cosine_sim_title.shape}')

   Title-only model
   Vocabulary size : 11,382 terms
   TF-IDF matrix   : (2782, 11382)
   Cosine sim mat  : (2782, 2782)


In [26]:
# ── Model 2: Enriched (Title + Subject + Level) ───────────────────────────────
tfidf_enriched = TfidfVectorizer(
    analyzer      = 'word',
    ngram_range   = (1, 2),
    min_df        = 1,
    stop_words    = 'english',
    sublinear_tf  = True
)

tfidf_matrix_enriched = tfidf_enriched.fit_transform(df['soup'])
cosine_sim_enriched   = cosine_similarity(tfidf_matrix_enriched, tfidf_matrix_enriched)

print(f'   Enriched model (title + subject + level)')
print(f'   Vocabulary size : {len(tfidf_enriched.vocabulary_):,} terms')
print(f'   TF-IDF matrix   : {tfidf_matrix_enriched.shape}')
print(f'   Cosine sim mat  : {cosine_sim_enriched.shape}')

   Enriched model (title + subject + level)
   Vocabulary size : 14,761 terms
   TF-IDF matrix   : (2782, 14761)
   Cosine sim mat  : (2782, 2782)


---
## 5.  Recommendation Functions

In [28]:
# Build reverse lookup: course_title (lowercase) → DataFrame index
indices = pd.Series(df.index, index=df['course_title'].str.lower()).drop_duplicates()


def get_recommendations(
    query,
    cosine_sim,
    tfidf_vectorizer,
    df,
    indices,
    top_n          = 10,
    filter_subject = None,
    filter_level   = None
):
    query_clean = clean_text(query)

    # ── Path A: exact title match ────────────────────────────────────────────
    if query.lower() in indices.index:
        raw = indices[query.lower()]
        idx = int(raw.iloc[0] if hasattr(raw, 'iloc') else raw)
        # .tolist() converts numpy floats → plain Python floats so sorted() works
        sim_scores = list(enumerate(cosine_sim[idx].tolist()))

    # ── Path B: free-text query ──────────────────────────────────────────────
    else:
        query_vec  = tfidf_vectorizer.transform([query_clean])
        tfidf_mat  = tfidf_vectorizer.transform(df['soup'])
        sim_array  = cosine_similarity(query_vec, tfidf_mat).flatten().tolist()
        sim_scores = list(enumerate(sim_array))

    # Sort by similarity descending — works now because all scores are plain floats
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Exclude the query course itself
    sim_scores = [(i, s) for i, s in sim_scores if s < 0.9999]

    results = []
    for idx, score in sim_scores:
        row = df.iloc[idx]
        if filter_subject and row['subject'] != filter_subject:
            continue
        if filter_level and row['level'] != filter_level:
            continue
        results.append({
            'course_title'    : row['course_title'],
            'subject'         : row['subject'],
            'level'           : row['level'],
            'price'           : row['price'],
            'num_subscribers' : row['num_subscribers'],
            'num_reviews'     : row['num_reviews'],
            'num_lectures'    : row['num_lectures'],
            'content_duration': row['content_duration'],
            'is_paid'         : row['is_paid'],
            'url'             : row['url'],
            'similarity_score': round(score, 4)
        })
        if len(results) == top_n:
            break

    return pd.DataFrame(results)


print(' Recommendation function defined')

 Recommendation function defined


---
## 6.  Test the Recommender

In [29]:
# ── Test 1: Exact title match ─────────────────────────────────────────────────
query = 'Ultimate Investment Banking Course'

print(f'🔎 Query: "{query}"')
print('─' * 70)
results = get_recommendations(
    query,
    cosine_sim      = cosine_sim_enriched,
    tfidf_vectorizer= tfidf_enriched,
    df              = df,
    indices         = indices,
    top_n           = 5
)
results[['course_title','subject','level','price','num_subscribers','similarity_score']]

🔎 Query: "Ultimate Investment Banking Course"
──────────────────────────────────────────────────────────────────────


,course_title,subject,level,price,num_subscribers,similarity_score
0,Advanced Accounting for Investment Banking,Business Finance,Intermediate Level,50,1260,0.3672
1,The Investment Banking Recruitment Series,Business Finance,All Levels,40,17,0.3433
2,Investment Banking: How to Land a Job on Wall ...,Business Finance,All Levels,75,1218,0.2810
3,The Ultimate jQuery Course,Web Development,All Levels,20,1098,0.2797
4,Investment Banking Operations : Securities Tra...,Business Finance,All Levels,30,267,0.2590


In [30]:
# ── Test 2: Free-text / keyword search ────────────────────────────────────────
query = 'python web development'

print(f' Query: "{query}"')
print('─' * 70)
results = get_recommendations(
    query,
    cosine_sim      = cosine_sim_enriched,
    tfidf_vectorizer= tfidf_enriched,
    df              = df,
    indices         = indices,
    top_n           = 5
)
results[['course_title','subject','level','price','num_subscribers','similarity_score']]

 Query: "python web development"
──────────────────────────────────────────────────────────────────────


,course_title,subject,level,price,num_subscribers,similarity_score
0,Python Web Programming,Web Development,Beginner Level,100,1020,0.6433
1,Advanced Scalable Python Web Development Using...,Web Development,Intermediate Level,120,1299,0.4489
2,Projects in Django and Python,Web Development,All Levels,60,1764,0.4129
3,Introduction to QGIS Python Programming,Web Development,Beginner Level,85,197,0.2366
4,Learn Python Django - A Hands-On Course,Web Development,Beginner Level,50,1339,0.2274


In [31]:
# ── Test 3: With subject + level filter ───────────────────────────────────────
query = 'guitar lessons for beginners'

print(f' Query: "{query}" | Filter: Musical Instruments + Beginner Level')
print('─' * 70)
results = get_recommendations(
    query,
    cosine_sim       = cosine_sim_enriched,
    tfidf_vectorizer = tfidf_enriched,
    df               = df,
    indices          = indices,
    top_n            = 5,
    filter_subject   = 'Musical Instruments',
    filter_level     = 'Beginner Level'
)
results[['course_title','subject','level','price','num_subscribers','similarity_score']]

 Query: "guitar lessons for beginners" | Filter: Musical Instruments + Beginner Level
──────────────────────────────────────────────────────────────────────


,course_title,subject,level,price,num_subscribers,similarity_score
0,Acoustic Guitar Lessons For Beginners,Musical Instruments,Beginner Level,95,10,0.6443
1,Bass Guitar Lessons For Beginners,Musical Instruments,Beginner Level,95,180,0.6343
2,Piano Lessons For Beginners,Musical Instruments,Beginner Level,95,8,0.4808
3,Beginner Guitar Lessons,Musical Instruments,Beginner Level,20,118,0.4580
4,Drum Lessons For Beginners,Musical Instruments,Beginner Level,95,14,0.4490


In [32]:
# ── Test 4: Graphic design keyword ────────────────────────────────────────────
query = 'logo design photoshop'

print(f' Query: "{query}"')
print('─' * 70)
results = get_recommendations(
    query,
    cosine_sim       = cosine_sim_enriched,
    tfidf_vectorizer = tfidf_enriched,
    df               = df,
    indices          = indices,
    top_n            = 5
)
results[['course_title','subject','level','price','num_subscribers','similarity_score']]

 Query: "logo design photoshop"
──────────────────────────────────────────────────────────────────────


,course_title,subject,level,price,num_subscribers,similarity_score
0,LOGO DESIGN IN POWERPOINT,Graphic Design,Beginner Level,95,1046,0.3995
1,Logo Design - Design a Logo in Photoshop for b...,Graphic Design,Beginner Level,30,740,0.3971
2,Logo Design for Entrepreneurs,Graphic Design,Beginner Level,20,274,0.3830
3,Marvelous Logo Design,Graphic Design,All Levels,20,91,0.3707
4,Illustrated Logo Design,Graphic Design,Intermediate Level,25,447,0.3657


---
## 7.  Quick Model Evaluation

Since this is an **unsupervised** recommendation task (no ground-truth labels),  
we evaluate using **intra-list similarity** and **subject consistency**.

In [34]:
def evaluate_subject_consistency(queries, cosine_sim, tfidf_vectorizer, df, indices, top_n=10):
    scores = []
    for query in queries:
        q_lower = query.lower()
        if q_lower not in indices.index:
            continue

        raw          = indices[q_lower]
        idx          = int(raw.iloc[0] if hasattr(raw, 'iloc') else raw)
        true_subject = df.iloc[idx]['subject']

        recs = get_recommendations(
                   query, cosine_sim, tfidf_vectorizer,
                   df, indices, top_n=top_n
               )
        if len(recs) == 0:
            continue
        pct = (recs['subject'] == true_subject).mean()
        scores.append(pct)
    return np.mean(scores) if scores else 0.0


# Sample 50 random courses as test queries
np.random.seed(42)
sample_queries = df['course_title'].sample(50).tolist()

score_title    = evaluate_subject_consistency(
    sample_queries, cosine_sim_title,    tfidf_title,    df, indices
)
score_enriched = evaluate_subject_consistency(
    sample_queries, cosine_sim_enriched, tfidf_enriched, df, indices
)

print('=== Subject Consistency Score (higher is better) ===')
print(f'  Title-only model : {score_title:.2%}')
print(f'  Enriched model   : {score_enriched:.2%}')
print()
print('→ Enriched model is selected for production.' if score_enriched >= score_title
      else '→ Title-only model is selected for production.')

=== Subject Consistency Score (higher is better) ===
  Title-only model : 87.00%
  Enriched model   : 97.40%

→ Enriched model is selected for production.


In [35]:
def avg_similarity_score(queries, cosine_sim, tfidf_vectorizer, df, indices, top_n=10):
    """
    Average cosine similarity score across top-N recommendations.
    Measures how 'close' the recommended courses are to the query.
    """
    all_scores = []
    for query in queries:
        recs = get_recommendations(
                   query, cosine_sim, tfidf_vectorizer,
                   df, indices, top_n=top_n
               )
        if len(recs) > 0:
            all_scores.extend(recs['similarity_score'].tolist())
    return np.mean(all_scores) if all_scores else 0.0


avg_title    = avg_similarity_score(
    sample_queries, cosine_sim_title,    tfidf_title,    df, indices
)
avg_enriched = avg_similarity_score(
    sample_queries, cosine_sim_enriched, tfidf_enriched, df, indices
)

print('=== Average Similarity Score (higher = tighter recommendations) ===')
print(f'  Title-only model : {avg_title:.4f}')
print(f'  Enriched model   : {avg_enriched:.4f}')

=== Average Similarity Score (higher = tighter recommendations) ===
  Title-only model : 0.2196
  Enriched model   : 0.2520


---
## 8.  Save Model Artifacts

We save everything needed by the Streamlit app into a single `.pkl` file:

In [ ]:
import os

artifacts = {
    # ── Enriched model (primary) ──────────────────────────────────────────
    'tfidf_enriched'      : tfidf_enriched,
    'cosine_sim_enriched' : cosine_sim_enriched,

    # ── Title-only model (fallback / comparison) ──────────────────────────
    'tfidf_title'         : tfidf_title,
    'cosine_sim_title'    : cosine_sim_title,

    # ── Data ──────────────────────────────────────────────────────────────
    'df'                  : df,
    'indices'             : indices,
}

output_path = 'recommender_artifacts.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(artifacts, f)

size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f' Artifacts saved → "{output_path}"  ({size_mb:.1f} MB)')
print()
print('Saved keys:')
for k in artifacts:
    print(f'   {k}')

 Artifacts saved → "recommender_artifacts.pkl"  (120.1 MB)

Saved keys:
  • tfidf_enriched
  • cosine_sim_enriched
  • tfidf_title
  • cosine_sim_title
  • df
  • indices


In [ ]:
# ── Interactive Model Test ────────────────────────────────────────────────────

def test_recommender(query, top_n=5, filter_subject=None, filter_level=None):
    """
    Quick test wrapper — prints a clean results table for any query.
    """
    print('=' * 70)
    print(f'  🔎 Query         : "{query}"')
    print(f'  📂 Subject filter : {filter_subject or "None"}')
    print(f'  🎓 Level filter   : {filter_level   or "None"}')
    print(f'  📊 Top-N          : {top_n}')
    print('=' * 70)

    recs = get_recommendations(
        query,
        cosine_sim       = cosine_sim_enriched,
        tfidf_vectorizer = tfidf_enriched,
        df               = df,
        indices          = indices,
        top_n            = top_n,
        filter_subject   = filter_subject,
        filter_level     = filter_level
    )

    if recs.empty:
        print('  ⚠️  No recommendations found. Try a different query or remove filters.')
        return

    for rank, (_, row) in enumerate(recs.iterrows(), start=1):
        price_str = 'Free' if row['price'] == 0 else ('$' + str(row['price']))
        print(f"\n  [{rank}] {row['course_title']}")
        print(f"       📁 Subject    : {row['subject']}")
        print(f"       🎓 Level      : {row['level']}")
        print(f"       💰 Price      : {price_str}")
        print(f"       👥 Subscribers: {row['num_subscribers']:,}")
        print(f"       ⭐ Reviews    : {row['num_reviews']:,}")
        print(f"       🎯 Similarity : {row['similarity_score']:.2%}")

    print('\n' + '=' * 70)


# ══════════════════════════════════════════════════════════════════════════════
# TEST CASES


# Test 1: Keyword search
test_recommender('Python for Web Development', top_n=2)
# Test 2: Exact title match
test_recommender('Ultimate Investment Banking Course', top_n=5)

# Test 3: Subject filter
test_recommender('guitar', top_n=5, filter_subject='Musical Instruments')

# Test 4: Subject + Level filter
test_recommender('design', top_n=5, filter_subject='Graphic Design', filter_level='Beginner Level')

# Test 5: Multi-keyword cross-domain
test_recommender('beginner finance excel', top_n=5)



  🔎 Query         : "Python for Web Development"
  📂 Subject filter : None
  🎓 Level filter   : None
  📊 Top-N          : 2

  [1] Python Web Programming
       📁 Subject    : Web Development
       🎓 Level      : Beginner Level
       💰 Price      : $100
       👥 Subscribers: 1,020
       ⭐ Reviews    : 46
       🎯 Similarity : 64.33%

  [2] Advanced Scalable Python Web Development Using Flask
       📁 Subject    : Web Development
       🎓 Level      : Intermediate Level
       💰 Price      : $120
       👥 Subscribers: 1,299
       ⭐ Reviews    : 56
       🎯 Similarity : 44.89%

  🔎 Query         : "Ultimate Investment Banking Course"
  📂 Subject filter : None
  🎓 Level filter   : None
  📊 Top-N          : 5

  [1] Advanced Accounting for Investment Banking
       📁 Subject    : Business Finance
       🎓 Level      : Intermediate Level
       💰 Price      : $50
       👥 Subscribers: 1,260
       ⭐ Reviews    : 80
       🎯 Similarity : 36.72%

  [2] The Investment Banking Recruitment S